# Notebook 04: Full End-to-End Fine-Tuning

In this notebook, we perform true end-to-end training. **All weights of the Moirai encoder and the classification heads are updated**.


In [13]:
import torch
import pandas as pd
from IPython.display import display

from heads import (
    MeanPoolingClassifier,
    SingleScaleAttentionClassifier,
    SingleScaleMultiHeadClassifier,
    HierarchicalMultiHeadClassifier,
)
from models import FullMaskOnlyWrapper, FullHeadWrapper
from utils import get_lsst_dataloaders, universal_grid_search

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES = [8, 16, 32, 64]

lr_grid = [5*1e-5]
wd_grid = [0.01, 0.05]

BATCH_SIZE = 256

In [14]:
tr_loader, va_loader, te_loader, num_classes = get_lsst_dataloaders(BATCH_SIZE, DEVICE)

In [15]:
METRICS_COLS = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted Precision", "Weighted Recall", "Weighted F1"]
df_patch_8_metrics = pd.DataFrame(columns=METRICS_COLS)
df_patch_8_metrics.index.name = "Model"

## 1. Baseline: Linear Model on Mask Embedding Only (Full FT)

In [16]:
df_mask_only = pd.DataFrame(index=["Mask Only"], columns=PATCH_SIZES)
df_mask_only.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    _, metrics = universal_grid_search(
        model_class=FullMaskOnlyWrapper,
        model_kwargs={"patch_size": patch, "num_vars": NUM_VARS, "num_classes": num_classes, "size": SIZE},
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid=wd_grid, lr_grid=lr_grid, verbose=True
    )
    df_mask_only.loc["Mask Only", patch] = metrics["Accuracy"]
    if patch == 8:
        df_patch_8_metrics.loc["Mask Only"] = metrics

LR=5e-05| WD=0.01
1.99163513067292
1.6944800072569188
1.5109288518021746
1.3895991682037105
1.3134967456988202
1.2564845104527667
1.2271249565651747
1.2311561291779929
1.2168133365429514
1.2088834929272412
1.2286442663611434
1.248081850811718
1.255764082195313
1.2748007628975846
1.308206302363698
 Early stopping : 14
Val Loss: 1.2089
LR=5e-05| WD=0.05
1.9659802758596776
1.7163413268763845
1.5288995514071084
1.4253127497386158
1.3445464750615561
1.3018999632781114
1.2637897613571911
1.252158387889707
1.2292636109561454
1.2396514270363785
1.2187249049907778
1.2586575833762563
1.2356862789247094
1.2399167200414145
1.2703512170450475
1.2672382554387658
 Early stopping : 15
Val Loss: 1.2187
LR=5e-05| WD=0.01
2.020311820797804
1.7466934570452062
1.5526140278917018
1.4413792961012057
1.354346293744033
1.3059315264709597
1.28584895967468
1.2581725857122157
1.2692044924914352
1.273002579929383
1.2968433698018391
1.3320042893169373
1.3385828181010921
 Early stopping : 12
Val Loss: 1.2582
LR=5e-0

In [17]:
print("Mask Only - Accuracy")
display(df_mask_only.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Mask Only - Accuracy


Patch Size,8,16,32,64
Mask Only,0.6172,0.6115,0.6046,0.6071



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.6172,0.4618,0.3446,0.3498,0.5675,0.6172,0.5657


## 2. Baseline: Mean Pooling

In [18]:
df_mean_pool = pd.DataFrame(index=["Mean Pooling"], columns=PATCH_SIZES)
df_mean_pool.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    _, metrics = universal_grid_search(
        model_class=FullHeadWrapper,
        model_kwargs={
            "head_class": MeanPoolingClassifier,
            "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
            "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid=wd_grid, lr_grid=lr_grid, verbose=True
    )
    df_mean_pool.loc["Mean Pooling", patch] = metrics["Accuracy"]
    if patch == 8:
        df_patch_8_metrics.loc["Mean Pooling"] = metrics

LR=5e-05| WD=0.01
1.98967732646601
1.6856873791392257
1.4975995174268397
1.376245954172398
1.2937133283149906
1.2265407806489526
1.188662949616347
1.146014923971843
1.1316695872361098
1.1057425989368097
1.1078848548051787
1.118649358671855
1.113788842185726
1.1160762251877203
1.1319558872439996
 Early stopping : 14
Val Loss: 1.1057
LR=5e-05| WD=0.05
2.0296942431752276
1.7406290498206285
1.5239129018008224
1.3922003304086081
1.3130336185780966
1.2288164065136173
1.1914966736382586
1.162642339380776
1.141283253344094
1.1275136645247297
1.1173694986638014
1.106620026797783
1.118584286875841
1.1210955972593974
1.1153169695923968
1.1819519802806824
1.2004383860564813
 Early stopping : 16
Val Loss: 1.1066
LR=5e-05| WD=0.01
2.0026083283308074
1.7317185014244016
1.535494533980765
1.3986073170250994
1.3037317593892415
1.2459205166111147
1.220081715079827
1.1790959457071817
1.1733618567629558
1.1922586206498185
1.234082970192762
1.1988577503498976
1.21543359174961
1.2639684240992477
 Early stopp

In [19]:
print("Mean Pooling - Accuracy")
display(df_mean_pool.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Mean Pooling - Accuracy


Patch Size,8,16,32,64
Mean Pooling,0.6318,0.6241,0.6298,0.631



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.6172,0.4618,0.3446,0.3498,0.5675,0.6172,0.5657
Mean Pooling,0.6318,0.4730,0.3573,0.3533,0.5702,0.6318,0.5624


## 3. Advanced Pooling: Attention & MHA (Full FT)

In [20]:
PATCH_SIZES_TO_TEST = [8, 16, 32, 64]
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 16

model_names_single = ["Attention (Base)", f"MHA (Heads={NUM_HEADS})"]
index_single = pd.MultiIndex.from_product([model_names_single, MODES], names=["Model", "Mode"])
df_adv_single = pd.DataFrame(index=index_single, columns=PATCH_SIZES_TO_TEST)
df_adv_single.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        # 1. Basic Attention
        _, metrics_attn = universal_grid_search(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleAttentionClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "mode": mode},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_adv_single.loc[("Attention (Base)", mode), patch] = metrics_attn["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"Attention ({mode})"] = metrics_attn

        # 2. Multi-Head Attention
        _, metrics_mha = universal_grid_search(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_adv_single.loc[(f"MHA (Heads={NUM_HEADS})", mode), patch] = metrics_mha["Accuracy"]

print("Attention & MHA - Accuracy")
display(df_adv_single.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

LR=5e-05| WD=0.01
 Early stopping : 15
Val Loss: 1.1589
LR=5e-05| WD=0.05
 Early stopping : 15
Val Loss: 1.1154
LR=5e-05| WD=0.01
 Early stopping : 15
Val Loss: 1.1249
LR=5e-05| WD=0.05


KeyboardInterrupt: 

## 4. Ablation Study: Number of Attention Heads

In [ ]:
HEADS_TO_TEST = [16, 32, 64, 128]

df_heads_ablation8 = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_heads_ablation8.index.name = "Mode"
df_heads_ablation8.columns.name = "Num Heads (Patch 8)"
df_heads_ablation16 = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_heads_ablation16.index.name = "Mode"
df_heads_ablation16.columns.name = "Num Heads (Patch 16)"

for PATCH in [8, 16]:
    for mode in MODES:
        for heads in HEADS_TO_TEST:
            _, metrics = universal_grid_search(
                model_class=FullHeadWrapper,
                model_kwargs={
                    "head_class": SingleScaleMultiHeadClassifier,
                    "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": heads},
                    "patch_size": PATCH, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
                },
                train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
                wd_grid=wd_grid, lr_grid=lr_grid, verbose=True
            )
            if PATCH == 8:
                df_heads_ablation8.loc[mode, heads] = metrics["Accuracy"]
                df_patch_8_metrics.loc[f"MHA-{heads} ({mode})"] = metrics
            else:
                df_heads_ablation16.loc[mode, heads] = metrics["Accuracy"]

LR=0.0001| WD=0.05
1.8792030946995184
1.587150946865237
1.3727774503754406
1.2533731005056117
1.2024532643760122
1.1295714349281498
1.1120942540285064
1.0980616110127146
1.1063881240239957
1.1157859486293018
1.1964236391269094
1.2842744298097564
1.4340674722097753
 Early stopping : 12
Val Loss: 1.0981
LR=0.0001| WD=0.05
1.9495068003491658
1.574629487060919
1.3791399632042987
1.3061113260625823
1.2286659430682174
1.1426450266101496
1.1761039224097398
1.1324723590680255
1.096756697670231
1.1422289600217246
1.1594579966087653
1.2534034320009433
1.2797474570390655
1.4661017792011664
 Early stopping : 13
Val Loss: 1.0968
LR=0.0001| WD=0.05
1.9351809577244083
1.554339984568154
1.35819765998096
1.2449165747417668
1.1559496265116747
1.1580696832842943
1.1797341573529128
1.1181464563540326
1.1252768272306861
1.187676118641365
1.1802993809304587
1.2340882464153011
1.3137529661984948
 Early stopping : 12
Val Loss: 1.1181
LR=0.0001| WD=0.05
1.9458519685559157
1.5944017637066725
1.3478356309053374


In [ ]:
print("Ablation: Num Heads (Patch = 8) - Accuracy")
display(df_heads_ablation8.astype(float).round(4))
print("Ablation: Num Heads (Patch = 16) - Accuracy")
display(df_heads_ablation16.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Ablation: Num Heads (Patch = 8) - Accuracy


Num Heads (Patch 8),16,32,64,128
Mode,,,,
shared_context,0.6237,0.6480,0.6277,0.6334
independent_context,0.6371,0.6436,0.6403,0.6391


Ablation: Num Heads (Patch = 16) - Accuracy


Num Heads (Patch 16),16,32,64,128
Mode,,,,
shared_context,0.6273,0.6310,0.6419,0.6290
independent_context,0.6277,0.6395,0.6375,0.6391



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.6196,0.4389,0.3398,0.3444,0.5618,0.6196,0.5624
Mean Pooling,0.6565,0.4276,0.3789,0.3802,0.5860,0.6565,0.6031
Attention (shared_context),0.6269,0.4200,0.3758,0.3693,0.5634,0.6269,0.5781
Attention (independent_context),0.6387,0.4777,0.3733,0.3738,0.5791,0.6387,0.5848
MHA-16 (shared_context),0.6237,0.3861,0.3634,0.3552,0.5479,0.6237,0.5652
MHA-32 (shared_context),0.6480,0.4111,0.3815,0.3790,0.5760,0.6480,0.5950
MHA-64 (shared_context),0.6277,0.4512,0.3739,0.3780,0.5675,0.6277,0.5691
MHA-128 (shared_context),0.6334,0.4648,0.3835,0.3834,0.5869,0.6334,0.5977
MHA-16 (independent_context),0.6371,0.4089,0.3724,0.3731,0.5707,0.6371,0.5851


## 5. Hierarchical MHA (Full FT)

In [ ]:
df_hierarchical = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
df_hierarchical.index.name = "Mode"
df_hierarchical.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        _, metrics_hier = universal_grid_search(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": HierarchicalMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_hierarchical.loc[mode, patch] = metrics_hier["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"Hierarchical ({mode})"] = metrics_hier

LR=0.0001| WD=0.05
 Early stopping : 15
Val Loss: 1.2529
LR=0.0001| WD=0.05
 Early stopping : 13
Val Loss: 1.2223
LR=0.0001| WD=0.05
 Early stopping : 13
Val Loss: 1.2558
LR=0.0001| WD=0.05
 Early stopping : 12
Val Loss: 1.2814
LR=0.0001| WD=0.05
 Early stopping : 12
Val Loss: 1.2924
LR=0.0001| WD=0.05
 Early stopping : 14
Val Loss: 1.2507
LR=0.0001| WD=0.05
 Early stopping : 13
Val Loss: 1.2847
LR=0.0001| WD=0.05
 Early stopping : 11
Val Loss: 1.2634


In [ ]:
print("Hierarchical MHA - Accuracy")
display(df_hierarchical.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Hierarchical MHA - Accuracy


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.6148,0.5994,0.5795,0.5876
independent_context,0.5973,0.5848,0.5904,0.5961



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.6196,0.4389,0.3398,0.3444,0.5618,0.6196,0.5624
Mean Pooling,0.6565,0.4276,0.3789,0.3802,0.5860,0.6565,0.6031
Attention (shared_context),0.6269,0.4200,0.3758,0.3693,0.5634,0.6269,0.5781
Attention (independent_context),0.6387,0.4777,0.3733,0.3738,0.5791,0.6387,0.5848
MHA-16 (shared_context),0.6237,0.3861,0.3634,0.3552,0.5479,0.6237,0.5652
MHA-32 (shared_context),0.6480,0.4111,0.3815,0.3790,0.5760,0.6480,0.5950
MHA-64 (shared_context),0.6277,0.4512,0.3739,0.3780,0.5675,0.6277,0.5691
MHA-128 (shared_context),0.6334,0.4648,0.3835,0.3834,0.5869,0.6334,0.5977
MHA-16 (independent_context),0.6371,0.4089,0.3724,0.3731,0.5707,0.6371,0.5851
